# **Evaluating Qwen3.5-4B for Diagnostic Recognition of Atypical Diseases: Endometriosis as a Case Study**

**This evaluation investigates whether Qwen3.5-4B is able to recognize endometriosis consistently across both typical (reproductive) and atypical (organ-specific and extrapelvic) presentations. It tests for recognition across disease type, investigates symptom bias, and conducts knowledge probes.**

**Install dependencies**

In [ ]:
!pip install -U transformers accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 69.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 13.1 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.18.0
    Uninstalling transformers-5.18.0:
      Successfully uninstalled transformers-5.18.0


**Set Up Qwen3.5-4B**

In [ ]:
import torch
from transformers import (
    AutoProcessor,
    AutoModelForMultimodalLM,
    BitsAndBytesConfig,
)

qwen_model_id = "Qwen/Qwen3.5-4B"

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,

)

qwen_processor = AutoProcessor.from_pretrained(qwen_model_id)

qwen_model = AutoModelForMultimodalLM.from_pretrained(
    qwen_model_id,
    quantization_config=quant_config,
    device_map="auto",
)

qwen_model.eval()

print("Qwen loaded successfully.")

preprocessor_config.json:   0%|          | 0.00/390 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/7.76k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/3.16k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/16.7k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 12.8MB            

...

model.safetensors.index.json:   0%|          | 0.00/76.2k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/723 [00:00<?, ?it/s]

Qwen loaded successfully.


In [ ]:
def run_qwen_case(vignette):
    messages = [
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": (
                        f"{vignette.strip()}\n\n"
                        "What is the most likely diagnosis? "
                        "Give one primary diagnosis and briefly explain your reasoning in no more than 3 sentences "
                        "based on the clinical presentation."
                    ),
                }
            ],
        }
    ]

    inputs = qwen_processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
        enable_thinking=False,
    ).to(qwen_model.device)

    input_len = inputs["input_ids"].shape[-1]

    with torch.inference_mode():
        generation = qwen_model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False,
        )

    output = qwen_processor.decode(
        generation[0][input_len:],
        skip_special_tokens=True
    ).strip()

    print("VIGNETTE:")
    print(vignette.strip())
    print("\nQWEN OUTPUT:")
    print(output)

    return output

# Experiment 1: **Presentation Type**



**The first experiment tests whether endometriosis recognition changes depending on whether the disease presents typically or atypically. There are 12 clinical vigenettes divided into three groups: pelvic and reproductive (typical), organ-specific (atypical), and extrapelvic (atypical).**


Reproductive and Pelvic Presentation (Typical)

In [ ]:
vignette = """
A 25-year-old woman presents with worsening painful menstruation,
pelvic pain, and pain during sexual intercourse.
"""

Q_C1_output = run_qwen_case(vignette)

[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.
[transformers] `causal_conv1d_update` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `fused_recurrent_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.


VIGNETTE:
A 25-year-old woman presents with worsening painful menstruation,
pelvic pain, and pain during sexual intercourse.

QWEN OUTPUT:
The most likely diagnosis is endometriosis, given the classic triad of worsening dysmenorrhea (painful menstruation), chronic pelvic pain, and dyspareunia (pain during intercourse). This condition involves the growth of endometrial-like tissue outside the uterus, which causes inflammation, adhesions, and deep infiltrating lesions that explain the specific pain patterns described. The patient's age and the progressive nature of her symptoms further support this diagnosis over other causes of pelvic pain.


In [ ]:
vignette = """
A 30-year-old woman presents with progressively worsening painful
menstruation and pain during sexual intercourse. She has also been
trying unsuccessfully to conceive.
"""

Q_C2_output = run_qwen_case(vignette)

VIGNETTE:
A 30-year-old woman presents with progressively worsening painful
menstruation and pain during sexual intercourse. She has also been
trying unsuccessfully to conceive.

QWEN OUTPUT:
The most likely diagnosis is endometriosis. This conclusion is based on the classic triad of symptoms presented: progressive dysmenorrhea (painful menstruation), dyspareunia (pain during intercourse), and infertility, which are hallmark clinical features of this condition. The worsening nature of the pain over time suggests the progressive accumulation of endometrial tissue outside the uterus, leading to inflammation and adhesions that impair fertility.


In [ ]:
vignette = """
A 37-year-old woman presents with chronic pelvic pain, painful
menstruation, pain during sexual intercourse, and difficulty
becoming pregnant.
"""

Q_C3_output = run_qwen_case(vignette)

VIGNETTE:
A 37-year-old woman presents with chronic pelvic pain, painful
menstruation, pain during sexual intercourse, and difficulty
becoming pregnant.

QWEN OUTPUT:
The most likely diagnosis is endometriosis. This condition is characterized by the presence of endometrial-like tissue outside the uterus, which causes inflammation, scarring, and adhesions that lead to the triad of chronic pelvic pain, dysmenorrhea (painful menstruation), and dyspareunia (pain during intercourse). Additionally, the formation of endometriomas or adhesions can distort pelvic anatomy and impair ovum transport, resulting in the difficulty conceiving described in this patient.


In [ ]:
vignette = """
A 29-year-old woman presents with infertility, painful menstruation,
and deep pain during sexual intercourse.
"""

Q_C4_output = run_qwen_case(vignette)

VIGNETTE:
A 29-year-old woman presents with infertility, painful menstruation,
and deep pain during sexual intercourse.

QWEN OUTPUT:
The most likely diagnosis is endometriosis. This conclusion is based on the classic triad of symptoms presented: dysmenorrhea (painful menstruation), dyspareunia (deep pain during intercourse), and infertility, which are hallmark clinical features of this condition. The deep pain specifically during intercourse suggests endometrial tissue implants in the uterosacral ligaments or deep pelvic structures, causing inflammation and adhesions that interfere with conception.


Organ-Specific Presentation (Atypical)

In [ ]:
vignette = """
A 41-year-old woman experiences recurrent rectal bleeding
that consistently occurs around the time of menstruation.
"""

Q_O1_output = run_qwen_case(vignette)

VIGNETTE:
A 41-year-old woman experiences recurrent rectal bleeding
that consistently occurs around the time of menstruation.

QWEN OUTPUT:
The most likely diagnosis is **intermenstrual hemorrhage** (specifically related to menstrual cycle fluctuations) or **endometriosis** involving the rectum. The reasoning is that the bleeding consistently coincides with menstruation, suggesting a hormonal or inflammatory process linked to the menstrual cycle rather than a sporadic structural lesion like a polyp or cancer. While endometriosis is a strong candidate given the cyclical nature, the specific pattern of bleeding around menses without other systemic symptoms could also represent benign hormonal-related mucosal changes or minor endometriosis affecting the rectal mucosa.


In [ ]:
vignette = """
A woman experiences recurrent painful bowel movements
that consistently occur during menstruation.
"""

Q_O2_output = run_qwen_case(vignette)

VIGNETTE:
A woman experiences recurrent painful bowel movements
that consistently occur during menstruation.

QWEN OUTPUT:
The most likely diagnosis is **endometriosis**, a condition where endometrial-like tissue grows outside the uterus and responds to hormonal cycles. This tissue bleeds during menstruation, causing inflammation, scarring, and intense pain in the bowel that coincides with the menstrual period. The specific pattern of pain occurring consistently only during menstruation strongly distinguishes this from other causes of chronic bowel pain.


In [ ]:
vignette = """
A 29-year-old woman experiences recurrent urinary frequency,
urgency, and pain when urinating that become worse during menstruation.
"""

Q_O3_output = run_qwen_case(vignette)

VIGNETTE:
A 29-year-old woman experiences recurrent urinary frequency,
urgency, and pain when urinating that become worse during menstruation.

QWEN OUTPUT:
The most likely diagnosis is **interstitial cystitis/hemorrhagic cystitis (IC/HC)**, also known as bladder pain syndrome. This condition is characterized by chronic bladder symptoms that are often exacerbated by hormonal fluctuations, such as the estrogen changes occurring during menstruation. The combination of frequency, urgency, and dysuria that specifically worsens with periods is a classic clinical presentation for this disorder.


In [ ]:
vignette = """
A 44-year-old woman experiences recurrent right-sided flank pain
that consistently becomes worse during menstruation.
"""

Q_O4_output = run_qwen_case(vignette)

VIGNETTE:
A 44-year-old woman experiences recurrent right-sided flank pain
that consistently becomes worse during menstruation.

QWEN OUTPUT:
The most likely diagnosis is **endometriosis**, specifically involving the right kidney or retroperitoneal structures. This condition is characterized by the presence of endometrial-like tissue outside the uterus, which responds to hormonal cycles by bleeding and causing inflammation, leading to pain that worsens during menstruation. The specific location of the pain in the right flank suggests that the ectopic endometrial implants are situated in the renal area or adjacent retroperitoneal tissues.


Extrapelvic Presentation (Atypical)

In [ ]:
vignette = """
A woman in her early 40s experiences recurrent right-sided chest pain
radiating to her shoulder that coincides with menstruation.
"""

Q_E1_output = run_qwen_case(vignette)

VIGNETTE:
A woman in her early 40s experiences recurrent right-sided chest pain
radiating to her shoulder that coincides with menstruation.

QWEN OUTPUT:
The most likely diagnosis is **gynecologic chest wall pain** (also known as menstrual-related chest wall pain or dysmenorrhea radiating to the chest). This condition is suggested because the patient is in her reproductive age, and the chest pain specifically coincides with her menstrual cycle, indicating a hormonal or visceral-referred origin rather than a primary cardiac issue. The radiation to the shoulder is a common pattern for referred pain from the diaphragm or intercostal muscles, which can be irritated by uterine contractions or prostaglandin surges during menstruation.


In [ ]:
vignette = """
A 44-year-old woman experiences recurrent sudden right-sided chest pain
and shortness of breath that begin around the start of menstruation.
"""

Q_E2_output = run_qwen_case(vignette)

VIGNETTE:
A 44-year-old woman experiences recurrent sudden right-sided chest pain
and shortness of breath that begin around the start of menstruation.

QWEN OUTPUT:
The most likely diagnosis is **spontaneous pneumothorax** (specifically secondary to a ruptured pulmonary bleb). This condition is strongly associated with the menstrual cycle due to hormonal fluctuations that cause alveolar wall weakening, leading to air leakage into the pleural space. The patient's age and the timing of symptoms coinciding with menstruation are classic indicators for this specific presentation.


In [ ]:
vignette = """
A 33-year-old woman with a previous cesarean-section scar experiences
recurrent pain and swelling at the scar that increase during menstruation.
"""

Q_E3_output = run_qwen_case(vignette)

VIGNETTE:
A 33-year-old woman with a previous cesarean-section scar experiences
recurrent pain and swelling at the scar that increase during menstruation.

QWEN OUTPUT:
The most likely diagnosis is a **scar endometriosis** (also known as an endometrioma of the cesarean scar). This condition is characterized by the presence of endometrial tissue within the uterine scar, which causes the specific symptoms of recurrent pain and swelling that exacerbate during menstruation due to cyclic bleeding. The patient's history of a prior cesarean section provides the necessary anatomical site for this ectopic endometrial growth to develop.


In [ ]:
vignette = """
A 34-year-old woman experiences recurrent episodes of coughing up blood
that begin during menstruation and resolve afterward.
"""

Q_E4_output = run_qwen_case(vignette)

VIGNETTE:
A 34-year-old woman experiences recurrent episodes of coughing up blood
that begin during menstruation and resolve afterward.

QWEN OUTPUT:
The most likely diagnosis is **menstrual hemoptysis** (also known as catamenial hemoptysis). This condition occurs when pulmonary veins rupture during menstruation due to increased vascularity and hormonal changes, causing blood to be coughed up rather than swallowed. The key diagnostic clue is the strict temporal relationship where the bleeding episodes begin during menstruation and resolve immediately afterward, distinguishing it from other causes of hemoptysis.


### **Results**

| Presentation Category | Correct Endometriosis Diagnoses |
|---|---:|
| Canonical pelvic/reproductive | 4 / 4 |
| Organ-specific pelvic | 2 / 4 |
| Extrapelvic | 1 / 4 |
| **Overall** | 7 / 12 |


Qwen3.5-4B recognized all classic reproductive presentations but struggled with organ-specific and extrapelvic presentations. This suggests that recognition of atypical presentation is challenging.

# Experiment 2: **Single Symptom Bias**

**To test for potential symptom bias, we feature three endometriosis cases as fixed baselines. Then, gastrintestinal, urinary, and an unrelated symptom are added separately:**

- baseline: endometriosis
- baseline + **diarrhea**: ?
- baseline + **urinary frequency**: ?
- baseline + **occasional tension headache**: ?

**Baseline 1**

In [ ]:
vignette = """
A 31-year-old woman experiences severe pain during menstruation
and deep pain during or after sexual intercourse.
"""

Q_B1_baseline = run_qwen_case(vignette)

VIGNETTE:
A 31-year-old woman experiences severe pain during menstruation
and deep pain during or after sexual intercourse.

QWEN OUTPUT:
The most likely diagnosis is endometriosis, given the combination of severe dysmenorrhea (painful menstruation) and deep dyspareunia (painful intercourse). These specific symptoms, particularly the deep pain during or after sex, are hallmark clinical indicators of endometrial tissue growing outside the uterus. While other conditions like adenomyosis can cause menstrual pain, the presence of deep dyspareunia strongly points toward endometriosis as the primary cause.


In [ ]:
vignette = """
A 31-year-old woman experiences severe pain during menstruation,
deep pain during or after sexual intercourse, and recurrent diarrhea.
"""

Q_B1_GI = run_qwen_case(vignette)

VIGNETTE:
A 31-year-old woman experiences severe pain during menstruation,
deep pain during or after sexual intercourse, and recurrent diarrhea.

QWEN OUTPUT:
The most likely diagnosis is **endometriosis**, a condition where endometrial-like tissue grows outside the uterus. This diagnosis is supported by the classic triad of symptoms: severe dysmenorrhea (painful periods), dyspareunia (deep pain during intercourse), and gastrointestinal disturbances like diarrhea caused by the inflammatory lesions affecting pelvic organs. While chronic pelvic pain can stem from other causes, the specific combination of deep dyspareunia and menstrual-related pain strongly points to this condition.


In [ ]:
vignette = """
A 31-year-old woman experiences severe pain during menstruation,
deep pain during or after sexual intercourse, and urinary frequency.
"""

Q_B1_urinary = run_qwen_case(vignette)

VIGNETTE:
A 31-year-old woman experiences severe pain during menstruation,
deep pain during or after sexual intercourse, and urinary frequency.

QWEN OUTPUT:
The most likely diagnosis is **endometriosis**, a condition where endometrial-like tissue grows outside the uterus. This diagnosis is supported by the classic triad of symptoms: severe dysmenorrhea (painful periods), dyspareunia (deep pain during intercourse), and urinary frequency caused by the presence of endometrial implants on the bladder or uterosacral ligaments. These specific symptoms together strongly distinguish the condition from other causes of pelvic pain such as simple fibroids or ovarian cysts.


In [ ]:
vignette = """
A 31-year-old woman experiences severe pain during menstruation,
deep pain during or after sexual intercourse, and occasional tension headaches.
"""

Q_B1_unrelated = run_qwen_case(vignette)

VIGNETTE:
A 31-year-old woman experiences severe pain during menstruation,
deep pain during or after sexual intercourse, and occasional tension headaches.

QWEN OUTPUT:
The most likely diagnosis is **endometriosis**, a condition where tissue similar to the uterine lining grows outside the uterus. This diagnosis is supported by the classic triad of symptoms presented: severe dysmenorrhea (painful periods), deep dyspareunia (painful intercourse), and extrapelvic pain manifesting as tension headaches. While the headaches can be associated with stress or migraines, the specific combination of deep pelvic pain during menstruation and intercourse is highly characteristic of endometrial implants infiltrating surrounding tissues.


**Baseline 2**

In [ ]:
vignette = """
A 25-year-old woman presents with worsening painful menstruation
and recurrent pelvic pain.
"""

Q_B2_baseline = run_qwen_case(vignette)

VIGNETTE:
A 25-year-old woman presents with worsening painful menstruation
and recurrent pelvic pain.

QWEN OUTPUT:
The most likely diagnosis is endometriosis, given the patient's age and the classic presentation of worsening dysmenorrhea (painful menstruation) accompanied by chronic pelvic pain. This condition is characterized by the presence of endometrial-like tissue outside the uterus, which causes inflammation and adhesions that intensify during menstruation. The recurrence of symptoms aligns with the cyclical nature of endometriosis, where pain typically escalates in the days leading up to and during menstruation.


In [ ]:
vignette = """
A 25-year-old woman presents with worsening painful menstruation,
recurrent pelvic pain, and recurrent diarrhea.
"""

Q_B2_GI = run_qwen_case(vignette)

VIGNETTE:
A 25-year-old woman presents with worsening painful menstruation,
recurrent pelvic pain, and recurrent diarrhea.

QWEN OUTPUT:
The most likely diagnosis is **endometriosis**, a condition where endometrial-like tissue grows outside the uterus and causes severe, cyclical pelvic pain and dysmenorrhea. The concurrent recurrent diarrhea is a common gastrointestinal manifestation caused by the inflammation and irritation of the bowel by the ectopic endometrial implants. This triad of symptoms in a young woman is highly characteristic of the disease, distinguishing it from other causes of chronic pelvic pain.


In [ ]:
vignette = """
A 25-year-old woman presents with worsening painful menstruation,
recurrent pelvic pain, and urinary frequency.
"""

Q_B2_urinary = run_qwen_case(vignette)

VIGNETTE:
A 25-year-old woman presents with worsening painful menstruation,
recurrent pelvic pain, and urinary frequency.

QWEN OUTPUT:
The most likely diagnosis is endometriosis, given the classic triad of worsening dysmenorrhea, chronic pelvic pain, and urinary frequency caused by endometrial tissue infiltrating the bladder or uterosacral ligaments. This condition typically presents in reproductive-aged women with symptoms that intensify during menstruation due to the cyclic shedding of ectopic tissue. The urinary frequency specifically suggests bladder involvement, which is a common extra-uterine manifestation of this disease.


In [ ]:
vignette = """
A 25-year-old woman presents with worsening painful menstruation,
recurrent pelvic pain, and occasional tension headaches.
"""

Q_B2_unrelated = run_qwen_case(vignette)

VIGNETTE:
A 25-year-old woman presents with worsening painful menstruation,
recurrent pelvic pain, and occasional tension headaches.

QWEN OUTPUT:
The most likely diagnosis is **endometriosis**, given the classic triad of worsening dysmenorrhea (painful menstruation), chronic pelvic pain, and associated tension headaches often seen in this condition. This diagnosis is supported by the patient's young age and the progressive nature of her symptoms, which are hallmark features distinguishing it from primary dysmenorrhea. While other conditions like adenomyosis or pelvic inflammatory disease are possible, the specific combination of severe menstrual pain and recurrent headaches in a 25-year-old strongly points toward endometriosis.


**Baseline 3**

In [ ]:
vignette = """
A 29-year-old woman presents with difficulty becoming pregnant
and painful menstruation.
"""

Q_B3_baseline = run_qwen_case(vignette)

VIGNETTE:
A 29-year-old woman presents with difficulty becoming pregnant
and painful menstruation.

QWEN OUTPUT:
The most likely diagnosis is endometriosis, a condition where tissue similar to the uterine lining grows outside the uterus. This ectopic tissue responds to hormonal changes during menstruation by bleeding and causing inflammation, which directly explains the patient's severe painful periods (dysmenorrhea). Additionally, the chronic inflammation and adhesions associated with endometriosis frequently interfere with ovulation or egg transport, leading to the difficulty in conceiving.


In [ ]:
vignette = """
A 29-year-old woman presents with difficulty becoming pregnant,
painful menstruation, and recurrent diarrhea.
"""

Q_B3_GI = run_qwen_case(vignette)

VIGNETTE:
A 29-year-old woman presents with difficulty becoming pregnant,
painful menstruation, and recurrent diarrhea.

QWEN OUTPUT:
The most likely diagnosis is **endometriosis**. This condition is characterized by the presence of endometrial tissue outside the uterus, which commonly causes severe dysmenorrhea (painful menstruation) and can lead to infertility due to adhesions or inflammation. The concurrent recurrent diarrhea suggests that the ectopic tissue may be located in the bowel or bladder, a frequent site for endometriosis that can mimic inflammatory bowel disease symptoms.


In [ ]:
vignette = """
A 29-year-old woman presents with difficulty becoming pregnant,
painful menstruation, and urinary frequency.
"""

Q_B3_urinary = run_qwen_case(vignette)

VIGNETTE:
A 29-year-old woman presents with difficulty becoming pregnant,
painful menstruation, and urinary frequency.

QWEN OUTPUT:
The most likely diagnosis is endometriosis. This condition is characterized by the presence of endometrial tissue outside the uterus, which commonly causes severe dysmenorrhea (painful menstruation) and chronic pelvic pain. The associated urinary frequency and difficulty conceiving are typical complications resulting from the inflammation, adhesions, and potential involvement of the bladder or reproductive organs seen in this disease.


In [ ]:
vignette = """
A 29-year-old woman presents with difficulty becoming pregnant,
painful menstruation, and occasional tension headaches.
"""

Q_B3_unrelated = run_qwen_case(vignette)

VIGNETTE:
A 29-year-old woman presents with difficulty becoming pregnant,
painful menstruation, and occasional tension headaches.

QWEN OUTPUT:
The most likely diagnosis is endometriosis, given the classic triad of chronic pelvic pain (dysmenorrhea), infertility, and the patient's young age. This condition is characterized by the presence of endometrial-like tissue outside the uterus, which causes inflammation, scarring, and adhesions that disrupt ovulation and implantation while intensifying menstrual cramps. The occasional tension headaches may be a secondary symptom related to chronic pain management or hormonal fluctuations associated with the disease.


**Results**

| Condition | Endometriosis Retained |
|---|---:|
| Baseline | 3 / 3 |
| + GI symptom | 3 / 3 |
| + Urinary symptom | 3 / 3 |
| + Unrelated symptom | 3 / 3 |


**Qwen3.5-4B remained stable when faced with additional symptoms across all three baselines. Adding a gastrointestinal, urinary, or unrelated symptom did not shift any of the three baseline diagnoses away from endometriosis.**

**This suggests that Qwen's failures with atypical presentations in Experiment 1 can't be simply explained by symptom bias.**

# Experiment 3: **Knowledge Probe**

**In experiment 3, the model is tested for explicit knowledge of atypical manifestations of endomteriosis.**

In [ ]:
def run_qwen_knowledge_probe(question):
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "text", "text": question.strip()}
            ],
        }
    ]

    inputs = qwen_processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
        enable_thinking=False
    ).to(qwen_model.device)

    input_len = inputs["input_ids"].shape[-1]

    with torch.inference_mode():
        generation = qwen_model.generate(
            **inputs,
            max_new_tokens=1024,
            do_sample=False,
        )

    output = qwen_processor.decode(
        generation[0][input_len:],
        skip_special_tokens=True
    ).strip()

    print("QUESTION:")
    print(question.strip())
    print("\nQWEN OUTPUT:")
    print(output)

    return output

**Endometriosis with Gastrointestinal Symptoms Knowledge Probe**

In [ ]:
Q_GI_knowledge = run_qwen_knowledge_probe("""
Can endometriosis cause gastrointestinal or bowel symptoms?

Briefly describe the gastrointestinal symptoms or manifestations that may occur.
""")

QUESTION:
Can endometriosis cause gastrointestinal or bowel symptoms?

Briefly describe the gastrointestinal symptoms or manifestations that may occur.

QWEN OUTPUT:
**Yes, endometriosis can absolutely cause gastrointestinal (GI) and bowel symptoms.**

While endometriosis is most commonly associated with pelvic pain and heavy menstrual bleeding, the disease frequently involves the digestive tract. This occurs when endometrial-like tissue grows outside the uterus and implants in the bowel wall, the peritoneum surrounding the intestines, or the mesentery. This tissue responds to hormonal changes during the menstrual cycle, leading to inflammation, scarring (adhesions), and physical distortion of the bowel.

### Common Gastrointestinal Manifestations

The symptoms typically follow a pattern where they **worsen during menstruation** and may improve afterward. Specific manifestations include:

*   **Cyclical Abdominal Pain:** Severe cramping or pain in the lower abdomen that intensifies jus

**Endometriosis with Urinary Symptoms Knowledge Probe**

In [ ]:
Q_urinary_knowledge = run_qwen_knowledge_probe("""
Can endometriosis cause urinary symptoms?

Briefly describe the urinary symptoms or manifestations that may occur.
""")

QUESTION:
Can endometriosis cause urinary symptoms?

Briefly describe the urinary symptoms or manifestations that may occur.

QWEN OUTPUT:
Yes, **endometriosis can absolutely cause urinary symptoms**. This occurs when endometrial tissue implants itself on the bladder, ureters, or surrounding pelvic structures, or when inflammation from the disease affects the urinary tract.

Here are the brief descriptions of the common urinary manifestations:

*   **Dysuria (Painful Urination)**: This is the most frequent symptom, often described as a burning sensation during urination. It can mimic a urinary tract infection (UTI) but is typically recurrent and linked to the menstrual cycle.
*   **Frequency and Urgency**: Patients may experience an increased need to urinate (frequency) or a sudden, compelling need to go (urgency), sometimes with only small amounts of urine passed.
*   **Hematuria (Blood in Urine)**: While less common than with other conditions, blood in the urine can occur if endometr

**Endometriosis with Extrapelvic Symptoms Knowledge Probe**

In [ ]:
Q_extrapelvic_knowledge = run_qwen_knowledge_probe("""
Can endometriosis occur outside the pelvic cavity?

Briefly describe examples of extrapelvic manifestations of endometriosis.
""")

QUESTION:
Can endometriosis occur outside the pelvic cavity?

Briefly describe examples of extrapelvic manifestations of endometriosis.

QWEN OUTPUT:
**Yes, endometriosis can occur outside the pelvic cavity.**

While the condition most commonly affects the ovaries, fallopian tubes, and the peritoneum within the pelvis, it can develop in virtually any site where endometrial-like tissue (glands and stroma) implants. This phenomenon is known as **extrapelvic endometriosis**. These implants can be discovered incidentally during surgery or imaging, or they may cause symptoms that mimic other conditions (such as chronic pain or infertility).

### Examples of Extrapelvic Manifestations

Endometriosis can manifest in various non-pelvic locations, including:

*   **Abdominal Cavity**: The most common extrapelvic sites include the **diaphragm** (often causing shoulder pain), the **peritoneum** of the abdomen, and the **omentum**.
*   **Gastrointestinal Tract**: It can implant in the **stomach**,

**Endometriosis with Thoracic Symptoms Knowledge Probe**

In [ ]:
Q_thoracic_knowledge = run_qwen_knowledge_probe("""
Can endometriosis affect the chest or thoracic cavity?

Briefly describe the thoracic symptoms or manifestations that may occur.
""")

QUESTION:
Can endometriosis affect the chest or thoracic cavity?

Briefly describe the thoracic symptoms or manifestations that may occur.

QWEN OUTPUT:
**Yes, endometriosis can affect the chest and thoracic cavity**, although it is less common than pelvic involvement. When endometrial tissue implants in the lungs, pleura, or mediastinum, it is referred to as **thoracic endometriosis**.

### Brief Description of Thoracic Symptoms and Manifestations

The clinical presentation varies significantly depending on whether the condition is **active** (during menstruation) or **inactive** (between periods).

*   **Cyclical Chest Pain**: The hallmark symptom is sharp, stabbing chest pain that occurs specifically during menstruation and subsides after the period ends. This is often mistaken for heart attacks or pulmonary embolisms.
*   **Hemoptysis (Coughing up blood)**: Active lesions can cause bleeding into the airways or pleural space, leading to coughing up blood.
*   **Pleuritic Pain**: Pai

### **Results**

| Knowledge Probe | Relevant Knowledge Demonstrated? |
|---|---:|
| Gastrointestinal / bowel manifestations | Yes |
| Urinary manifestations | Yes |
| Extrapelvic manifestations | Yes |
| Thoracic manifestations | Yes |



**As shown above, Qwen3.5-4B demonstrated explicit knowledge of atypical endometriosis. It acknowledged that it can manifest through gastrointestinal, urinary, extrapelvic, and thoracic symptoms.**
